Benötigte Module importieren und Datei laden. Die ersten Zeilen werden ausgegeben.

In [1]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

path = "../Data/iris.csv"
data = pd.read_csv(path, delimiter=',')

In [2]:
# Ausgabe der Korrelationen
correlations = data[data.columns].corr(numeric_only=True)
print('All correlations')
print('-' * 30)
correlations_abs_sum = correlations[correlations.columns].abs().sum()
print(correlations_abs_sum)
print('Weakest correlations')
print('-' * 30)
print(correlations_abs_sum.nsmallest(5))

All correlations
------------------------------
sepal.length    2.807265
sepal.width     1.912136
petal.length    3.263059
petal.width     3.146932
dtype: float64
Weakest correlations
------------------------------
sepal.width     1.912136
sepal.length    2.807265
petal.width     3.146932
petal.length    3.263059
dtype: float64


In [3]:
# species soll vorhergesagt werden
oe = OneHotEncoder()
col = oe.fit_transform(data[['species']])
col = col.toarray()
data = data.drop(['species'], axis = 1)

# Erzeuge Objekt
s_scaler = StandardScaler()
# Spalten für StandardScaler
data = s_scaler.fit_transform(data)

Daten vorbereiten.

KNN aufbauen

In [4]:
# Aus den zwei Tabellen vier Tabellen erzeugen
train_data, test_data, train_col, test_col = train_test_split(data,col, test_size=0.2, random_state=42)

# Aufbau KNN
model = tf.keras.Sequential()
model.add(tf.keras.Input(shape=(data.shape[1],)))
model.add(tf.keras.layers.Dense(32, activation=tf.nn.sigmoid))
model.add(tf.keras.layers.Dense(64, activation=tf.nn.sigmoid))
# Fix, sonst kommt ein Fehler beim Laden des Modells
#model.add(tf.keras.layers.Dense(3, activation=tf.nn.sigmoid.softmax))
model.add(tf.keras.layers.Dense(3, activation="softmax"))

# Konfiguration des Lernprozesses
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

Trainieren

In [5]:
# 70 Durchläufe
model.fit(train_data, train_col, epochs=70)

Epoch 1/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3585 - loss: 1.1223  
Epoch 2/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step - accuracy: 0.2919 - loss: 1.1119  
Epoch 3/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2987 - loss: 1.0922 
Epoch 4/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6331 - loss: 1.0824 
Epoch 5/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7302 - loss: 1.0719 
Epoch 6/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7827 - loss: 1.0633 
Epoch 7/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6652 - loss: 1.0510 
Epoch 8/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7912 - loss: 1.0394 
Epoch 9/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7688 - loss: 1.0310 
Epoch 10/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7937 - loss: 1.0184 
Epoch 11/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7740 - loss: 1.0080
Epoch 12/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7867 - loss: 0.9950 


Testen

In [6]:
test_loss, test_acc = model.evaluate(test_data, test_col)
print('Test accuracy:', test_acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.9333 - loss: 0.3212
Test accuracy: 0.9333333373069763


In [7]:
# KNN, Scaler und LabelEncoder speichern
model.save('model2.keras')
joblib.dump(s_scaler,'scaler2.joblib')
joblib.dump(oe,'oe2.joblib')

['oe2.joblib']